# SMS Spam Detection

This is an NLP classification project.

The file is a list of SMS messages. Each row is either ham, a normal message, or spam. Most of the rows are ham, so the spam class is the one that matters.

I will clean the text and train two models on the same split. The first is Naive Bayes on word counts, the usual baseline for short messages. The second is a Bidirectional LSTM with a learned embedding. Word order in a short SMS still carries the spam pattern, so the deep model is the one I want here. I will compare accuracy and F1, plot the confusion matrices, and save the LSTM.


In [ ]:
import kagglehub

path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")
print("Path to dataset files:", path)


In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = glob.glob(os.path.join(path, "**", "spam.csv"), recursive=True)[0]
df = pd.read_csv(csv_path, encoding="latin-1")
df = df[["v1", "v2"]]
df.columns = ["label", "text"]

df["text"] = df["text"].str.lower()
df["text"] = df["text"].str.replace(r"[^\w\s]", " ", regex=True)
df["text"] = df["text"].str.replace(r"\d+", " ", regex=True)
df["length"] = df["text"].str.split().str.len()

print(df.shape)
print(df["label"].value_counts())
print(df.head())

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.countplot(data=df, x="label")
plt.title("Class counts")
plt.subplot(1, 2, 2)
sns.histplot(data=df, x="length", hue="label", bins=40)
plt.title("Words per message")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

x = df["text"]
y = df["label"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.20, random_state=42, stratify=y
)

nb = Pipeline([
    ("vect", CountVectorizer(stop_words="english")),
    ("model", MultinomialNB()),
])
nb.fit(x_train, y_train)
nb_pred = nb.predict(x_test)

print("Naive Bayes")
print(classification_report(y_test, nb_pred))


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder

tf.keras.utils.set_random_seed(42)

le = LabelEncoder()
y_tr = le.fit_transform(y_train)
y_te = le.transform(y_test)
n_ham = (y_tr == 0).sum()
n_spam = (y_tr == 1).sum()

vectorize = layers.TextVectorization(max_tokens=8000, output_sequence_length=80)
vectorize.adapt(x_train.to_numpy())

dl = tf.keras.Sequential([
    vectorize,
    layers.Embedding(8000, 64, mask_zero=True),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])
dl.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
hist = dl.fit(
    x_train.to_numpy(), y_tr,
    validation_split=0.1,
    epochs=8,
    batch_size=32,
    class_weight={0: 1.0, 1: n_ham / n_spam},
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
    verbose=1,
)

dl_pred = (dl.predict(x_test.to_numpy(), verbose=0).ravel() >= 0.5).astype(int)
print("Bidirectional LSTM")
print(classification_report(y_te, dl_pred, target_names=le.classes_))


In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="val")
plt.title("LSTM accuracy")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(hist.history["loss"], label="train")
plt.plot(hist.history["val_loss"], label="val")
plt.title("LSTM loss")
plt.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, nb_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Naive Bayes")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
sns.heatmap(confusion_matrix(y_te, dl_pred), annot=True, fmt="d", cmap="Blues", ax=axes[1])
axes[1].set_title("Bidirectional LSTM")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("Actual")
plt.tight_layout()
plt.show()

print("Naive Bayes  accuracy", accuracy_score(y_test, nb_pred), "F1", f1_score(y_test, nb_pred, pos_label="spam"))
print("LSTM         accuracy", accuracy_score(y_te, dl_pred), "F1", f1_score(y_te, dl_pred, pos_label=1))


In [ ]:
samples = np.array([
    "hello are you coming to class tonight",
    "congratulations you won a free prize call this number now",
])
print("Naive Bayes", nb.predict(samples))
print("LSTM       ", le.inverse_transform((dl.predict(samples, verbose=0).ravel() >= 0.5).astype(int)))


In [ ]:
dl.save("sms_spam.keras")


# Conclusion

I cleaned the SMS text and trained Naive Bayes and a Bidirectional LSTM on the same 80/20 split. Naive Bayes on word counts is a strong baseline for short messages. The LSTM learns an embedding and reads the words in both directions, which is the deep learning model for this task. Most messages are ham, so spam F1 is the number to watch, not overall accuracy. The comparison cell prints both scores and the two confusion matrices. The LSTM is saved as sms_spam.keras.
